# Fluo_Analyse — Extractions informations & Comptage de lysosomes par image de microscopie

Ce notebook réalise l'**analyse en production** d'un lot d'images de microscopie
de fluorescence TIFF. Il s'appuie sur le CNN soma (`SomaCNN`) entraîné dans
`Pipeline_ML.ipynb` pour détecter les somas, puis utilise **Cellpose**
pour segmenter et compter les lysosomes à l'intérieur de chaque soma détecté.

**Pipeline par image :**
1. Sélection automatique du plan de meilleure netteté (z-stack)
2. Extraction des statistiques de fluorescence brutes
3. Suppression de la contribution des somas → statistiques hors-soma
4. Détection des somas (CNN + clustering)
5. Comptage des lysosomes dans chaque soma (Cellpose)
6. Export des résultats (images annotées + CSV)

**Dépendances :** classes et fonctions définies dans `Include.ipynb`.


## 1. Chargement des dépendances (`Include.ipynb`)

Charge toutes les bibliothèques, classes et fonctions utilitaires :
`SomaCNN`, `Best_plan_determination`, `CNN_Patches_Construction`,
`Clusterization_detected_patches`, `Finale_Fusion_patches`,
`Cellpose_Analyse_Count`, `Extract_fluo_informations`,
`Extract_fluo_informations_without_nucleus`, `Save_images_and_overlay`,
`Save_final_patches`, `Creation_CSV`, etc.


In [ ]:
%run "../Include.ipynb"


## 2. Vérification des versions et du GPU

Contrôle que Cellpose et PyTorch sont correctement installés,
et détecte la disponibilité d'un GPU CUDA.

> Le pipeline fonctionne sur CPU mais l'inférence Cellpose et CNN
> est significativement plus rapide sur GPU (×5 à ×20 selon la carte).


In [ ]:
from importlib.metadata import version
import torch

# Vérification version Cellpose
print("Cellpose version :", version("cellpose"))

# Vérification PyTorch et disponibilité GPU
print("PyTorch version  :", torch.__version__)
print("CUDA disponible  :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Nom du GPU       :", torch.cuda.get_device_name(0))
    print("Nombre de GPUs   :", torch.cuda.device_count())


## 3. Configuration des paramètres d'analyse

Tous les paramètres ajustables sont centralisés ici pour faciliter
la reproductibilité et les expériences comparatives.

### Paramètres CNN soma
| Paramètre | Description |
|---|---|
| `cnn_threshold` | Seuil de probabilité CNN pour retenir un patch comme soma (🔧 à calibrer selon le modèle) |
| `patch_size` | Taille des patches en pixels (doit correspondre à l'entraînement) |
| `overlap_thresh` | Fraction de chevauchement max entre deux ROI avant fusion (0.9 = 90 %) |

### Paramètres Cellpose (lysosomes)
| Paramètre | Description |
|---|---|
| `diameter` | Diamètre estimé des lysosomes en pixels |
| `cellprob_threshold` | Seuil de probabilité Cellpose (↓ = plus de détections) |
| `flow_threshold` | Cohérence des flux optiques (↑ = segmentation plus stricte) |
| `min/max_area` | Filtre morphologique : aire min/max des objets conservés (px²) |
| `min_circularity` | Filtre morphologique : circularité minimale (1 = cercle parfait) |
| `max_axis_ratio` | Filtre morphologique : élimine les structures allongées (dendrites) |
| `top_hat_radius` | Rayon du disque structural top-hat (doit être > rayon lysosome) |


In [ ]:
from cellpose import models
import torch
from pathlib import Path

# ── Dossier source des images TIFF ────────────────────────────────────────────
# 🔧 Modifier ce chemin pour pointer vers le dossier à analyser
dossier = Path(r'D:\ICS\DM5000 63X Mixtes n=2\20251031- Mixtes DIV 15\Ctrl\Ctrl1_TubulinA594 + pTau488')

# ── Paramètres de prétraitement image ─────────────────────────────────────────
pretraitement_sigma  = 1.5   # sigma du flou gaussien pré-détection
clip_limit           = 0.02  # limite CLAHE pour l'égalisation du contraste
distance_transform   = 40    # seuil de distance pour isoler le cœur du soma
margin               = 40    # marge (px) autour du soma pour la suppression

# ── Chargement du CNN soma ─────────────────────────────────────────────────────
# Détection automatique GPU/CPU
device    = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model_cnn = SomaCNN()
model_cnn.load_state_dict(torch.load("../soma_cnn_test.pth", map_location=device))
model_cnn = model_cnn.to(device)
model_cnn.eval()  # mode inférence (désactive dropout/batchnorm)

# ── Paramètres CNN patches ────────────────────────────────────────────────────
cnn_threshold = 0.23   # 🔧 seuil de probabilité CNN (calibré via courbe PR)
patch_size    = 64     # doit correspondre à la valeur utilisée à l'entraînement
half          = patch_size // 2
overlap_thresh = 0.9   # seuil de chevauchement pour la fusion des ROI

# ── Paramètres Cellpose (segmentation lysosomes) ──────────────────────────────
diameter           = 8     # 🔧 diamètre estimé des lysosomes (px)
cellprob_threshold = 0.1   # seuil de probabilité Cellpose
flow_threshold     = 0.5   # cohérence des flux optiques
min_area           = 5     # 🔧 aire minimale lysosome (px²)
max_area           = 500   # 🔧 aire maximale lysosome (px²)
min_circularity    = 0.5   # circularité minimale (filtre artefacts allongés)
max_axis_ratio     = 2     # rapport axes max (filtre dendrites)
top_hat_radius     = 10    # rayon disque top-hat (> rayon lysosome attendu)

# ── Initialisation du modèle Cellpose (cyto3) ─────────────────────────────────
# cyto3 : modèle généraliste recommandé pour les structures cytoplasmiques
cellpose_model = models.Cellpose(gpu=torch.cuda.is_available(), model_type='cyto3')

if torch.cuda.is_available():
    print("✅ GPU détecté :", torch.cuda.get_device_name(0))
else:
    print("⚠️  GPU non disponible → CPU utilisé")


## 4. Analyse en lot (`batch`)

Traite automatiquement tous les fichiers `.tif` du dossier (limité aux 10 premiers,
modifiable). Pour chaque image, le pipeline complet est exécuté :

1. **`Best_plan_determination`** : sélectionne le plan le plus net du z-stack
   (maximisation de la variance)
2. **`Extract_fluo_informations`** : statistiques brutes (aire, moyenne, min, max)
3. **`Extract_fluo_informations_without_nucleus`** : masque les somas et recalcule
   les statistiques sur les pixels restants (signal hors-soma)
4. **`CNN_Patches_Construction`** → **`Clusterization_detected_patches`** →
   **`Finale_Fusion_patches`** : détection et fusion des ROI somas
5. **`Cellpose_Analyse_Count`** : segmentation et comptage des lysosomes dans chaque ROI
6. **`Save_images_and_overlay`** / **`Save_final_patches`** : sauvegarde des images annotées
7. **`Creation_CSV`** : export de tous les résultats dans un CSV

> Pour activer la visualisation de débogage par patch,
> passer le dernier argument de `Cellpose_Analyse_Count` à `True`.


In [ ]:
# ── Dossier de sortie (créé automatiquement s'il n'existe pas) ───────────────
output_folder = dossier / "output"
output_folder.mkdir(exist_ok=True)

# ── Sélection des fichiers TIFF à traiter ────────────────────────────────────
# 🔧 Retirer [:10] pour traiter la totalité des fichiers du dossier
fichiers = list(dossier.glob("*.tif"))[:10]
results  = []

for fichier in tqdm(fichiers, desc="Traitement des fichiers", unit="fichier"):
    chemin = dossier / fichier.name

    # ── Étape 1 : sélection du meilleur plan z ───────────────────────────────
    img_best = Best_plan_determination(chemin, output_flag=False)

    # ── Étape 2 : statistiques de fluorescence brutes ────────────────────────
    area, pmean, pmin, pmax = Extract_fluo_informations(img_best)

    # ── Étape 3 : statistiques hors-soma (signal cytoplasmique) ─────────────
    img_true, mask, area_true, pmean_true, pmin_true, pmax_true = \
        Extract_fluo_informations_without_nucleus(
            img_best, distance_transform, margin, flag_visu=False
        )

    # Sauvegarde image originale + overlay du masque soma
    Save_images_and_overlay(output_folder, fichier, img_best, mask)

    # ── Étape 4 : détection des somas par CNN ────────────────────────────────
    all_probs, soma_patches = CNN_Patches_Construction(
        img_best, pretraitement_sigma, clip_limit,
        patch_size, cnn_threshold, model_cnn
    )
    # Fusion des patches qui se chevauchent en ROI uniques
    merged_rois             = Clusterization_detected_patches(img_best, soma_patches, patch_size)
    final_patches, filtered_rois = Finale_Fusion_patches(img_best, merged_rois, overlap_thresh)

    # ── Étape 5 : comptage des lysosomes par Cellpose ────────────────────────
    # Mettre flag_visu=True pour afficher la segmentation de chaque patch
    total_lysosomes = Cellpose_Analyse_Count(
        cellpose_model, final_patches,
        diameter, cellprob_threshold, flow_threshold,
        min_area, max_area, min_circularity, max_axis_ratio,
        top_hat_radius, flag_visu=False
    )

    # Sauvegarde image globale avec ROI annotées et probabilités CNN
    Save_final_patches(img_best, filtered_rois, soma_patches,
                       output_folder, fichier.name, half)

    # ── Étape 6 : agrégation des résultats ───────────────────────────────────
    results.append({
        "file"       : fichier.name,
        "area"       : area,         # aire totale image (px)
        "pmean"      : pmean,        # intensité moyenne brute
        "pmin"       : pmin,         # intensité min brute
        "pmax"       : pmax,         # intensité max brute
        "area_true"  : area_true,    # aire hors-soma (px)
        "pmean_true" : pmean_true,   # intensité moyenne hors-soma
        "pmin_true"  : pmin_true,    # intensité min hors-soma
        "pmax_true"  : pmax_true,    # intensité max hors-soma
        "n_lyso"     : total_lysosomes  # nombre de lysosomes détectés
    })

# ── Export CSV ────────────────────────────────────────────────────────────────
# Séparateur ";" et décimale "," pour compatibilité Excel (français)
Creation_CSV(results, output_folder)


## 5. Analyse d'un fichier individuel (mode débogage)

Permet d'inspecter visuellement le résultat du pipeline sur une image unique,
avec affichage activé (`output_flag=True`, `flag_visu=True`).

Utile pour :
- Vérifier la qualité de la sélection du plan z
- Contrôler le masque de suppression des somas avant le comptage
- Contrôler les différentes phases d'avancement du code via la visu

> Modifier `fichier` pour pointer vers l'image souhaitée.


In [ ]:
# ── Paramètres spécifiques à l'analyse individuelle ─────────────────────────
# 🔧 Modifier le chemin et le nom de fichier
dossier = Path(r'D:\ICS\DM5000 63X Mixtes n=2\20251031- Mixtes DIV 15\Ctrl\Ctrl1_TubulinA594 + pTau488')

output_folder = dossier / "output"
output_folder.mkdir(exist_ok=True)

# 🔧 Nom du fichier à analyser individuellement
fichier = "Ctrl1_Tubulin+pTau_001_w1TRITC.tif"
#fichier = "Ctrl1_Tubulin+pTau_019_w2GFP.tif"  # alternative canal GFP

chemin = dossier / fichier

# ── Étape 1 : sélection du meilleur plan z (avec visualisation) ──────────────
# output_flag=True : affiche tous les plans z avec leurs scores de netteté
img_best = Best_plan_determination(chemin, True)

# ── Étape 2 : statistiques de fluorescence brutes ─────────────────────────────
area, pmean, pmin, pmax = Extract_fluo_informations(img_best)
print(f"Aire totale   : {area} px")
print(f"Intensité moy : {pmean:.2f}")
print(f"Intensité min : {pmin} | max : {pmax}")

# ── Étape 3 : suppression soma + statistiques hors-soma ──────────────────────
# flag_visu=True : affiche l'image originale, le masque et l'image sans soma
img_true, mask, area_true, pmean_true, pmin_true, pmax_true = \
    Extract_fluo_informations_without_nucleus(
        img_best, distance_transform, margin, flag_visu=True
    )
print(f"\nAire hors-soma   : {area_true} px")
print(f"Intensité moy hs : {pmean_true:.2f}")
print(f"Intensité min hs : {pmin_true} | max hs : {pmax_true}")

# ── Étape 4 : détection des somas par CNN ────────────────────────────────
all_probs, soma_patches = CNN_Patches_Construction(img_best, pretraitement_sigma, clip_limit, patch_size, cnn_threshold, model_cnn)
# Fusion des patches qui se chevauchent en ROI uniques
merged_rois = Clusterization_detected_patches(img_best, soma_patches, patch_size)
final_patches, filtered_rois = Finale_Fusion_patches(img_best, merged_rois, overlap_thresh)

# Debug et affichage étapes intermédiaires
Plot_clusterised_patches(img_best, merged_rois)
Plot_final_patches(img_best, filtered_rois, soma_patches)
Plot_final_individual_patches(final_patches, filtered_rois)

# ── Étape 5 : comptage des lysosomes par Cellpose ────────────────────────
total_lysosomes = Cellpose_Analyse_Count(cellpose_model,
                                         final_patches,
                                         diameter, 
                                         cellprob_threshold,
                                         flow_threshold,
                                         min_area,
                                         max_area,
                                         min_circularity,
                                         max_axis_ratio,
                                         top_hat_radius,
                                         True) #Visualisation patch
